# Qwen3-1.7B TurboQuant 简短对比

这个 notebook 分成两部分：

1. 短输入：直接比较输出文本
2. 长输入：按 `prefill + gen` 方式比较耗时和显存

长输入部分和 `benchmark_turboquant.py` 的测法保持一致，避免 `generate()` 路径带来额外偏差。

In [1]:
import gc
import time

import torch
from transformers import AutoTokenizer

from my_qwen3 import Qwen3Config, Qwen3ForCausalLM

local_model_path = r"C:\Users\lihaodong\.cache\modelscope\hub\models\Qwen\Qwen3-1___7B"
tokenizer = AutoTokenizer.from_pretrained(local_model_path, trust_remote_code=True)

def build_inputs(prompt, enable_thinking=False):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    return tokenizer([text], return_tensors="pt")

def build_long_prompt(min_tokens=4096):
    seed = "请详细分析 TurboQuant 在 KV cache 压缩、长上下文推理、显存优化和 attention 计算中的关键工程权衡。"
    parts = []
    token_count = 0
    while token_count < min_tokens:
        parts.append(seed)
        token_count = len(tokenizer("".join(parts), add_special_tokens=False).input_ids)
    return "".join(parts), token_count

def load_model(config_override):
    config = Qwen3Config.from_pretrained(local_model_path)
    for key, value in config_override.items():
        setattr(config, key, value)
    return Qwen3ForCausalLM.from_pretrained(
        local_model_path,
        config=config,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

def run_short_case(description, config_override, prompt, max_new_tokens=128):
    model = load_model(config_override)
    inputs = build_inputs(prompt)
    input_ids = inputs.input_ids.to(model.device)
    attention_mask = inputs.attention_mask.to(model.device)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )
    elapsed = time.time() - start
    peak_mem_mb = torch.cuda.max_memory_allocated() / 1024**2

    new_tokens = outputs[0, input_ids.shape[1]:]
    output_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    del outputs
    del input_ids
    del attention_mask
    del inputs
    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "description": description,
        "output_text": output_text,
        "time_sec": elapsed,
        "peak_mem_mb": peak_mem_mb,
    }

def run_long_case(description, config_override, prompt, max_new_tokens=8):
    model = load_model(config_override)
    inputs = build_inputs(prompt)
    input_ids = inputs.input_ids.to(model.device)
    attention_mask = inputs.attention_mask.to(model.device)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    prefill_start = time.time()
    with torch.no_grad():
        prefill_outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=True,
        )
    prefill_time = time.time() - prefill_start
    prefill_mem_mb = torch.cuda.memory_allocated() / 1024**2
    prefill_peak_mb = torch.cuda.max_memory_allocated() / 1024**2

    past_key_values = prefill_outputs.past_key_values
    next_token = prefill_outputs.logits[:, -1:, :].argmax(dim=-1)

    gen_start = time.time()
    gen_peak_mb = prefill_peak_mb
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(
                input_ids=next_token,
                attention_mask=torch.ones(
                    (input_ids.shape[0], past_key_values.get_seq_length() + 1),
                    dtype=attention_mask.dtype,
                    device=model.device,
                ),
                past_key_values=past_key_values,
                use_cache=True,
            )
        past_key_values = outputs.past_key_values
        next_token = outputs.logits[:, -1:, :].argmax(dim=-1)
        gen_peak_mb = max(gen_peak_mb, torch.cuda.max_memory_allocated() / 1024**2)
    gen_time = time.time() - gen_start
    end_mem_mb = torch.cuda.memory_allocated() / 1024**2

    del outputs
    del prefill_outputs
    del past_key_values
    del input_ids
    del attention_mask
    del inputs
    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "description": description,
        "prefill_time_sec": prefill_time,
        "gen_time_sec": gen_time,
        "prefill_mem_mb": prefill_mem_mb,
        "peak_mem_mb": gen_peak_mb,
        "end_mem_mb": end_mem_mb,
    }


In [2]:
short_prompt = "请阅读下面这段话，并用中文总结其核心观点，同时给出 3 条简明结论：\n\nTurboQuant 通过 MSE + QJL 两阶段量化，对 KV cache 进行压缩，并尽量在压缩表示上直接完成 attention 计算，以降低长上下文推理中的显存占用。"

baseline_short = run_short_case("Baseline fp16 KV", {"use_turboquant": False}, short_prompt)
turbo_short = run_short_case(
    "TurboQuant 3-bit KV",
    {"use_turboquant": True, "turboquant_bits": 3, "turboquant_block_size": 256},
    short_prompt,
)

print("=" * 100)
for result in [baseline_short, turbo_short]:
    print(f"配置: {result['description']}")
    print(f"耗时: {result['time_sec']:.2f}s")
    print(f"峰值显存: {result['peak_mem_mb']:.2f} MB")
    print("输出文本:")
    print(result['output_text'])
    print("-" * 100)
print("=" * 100)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3ForCausalLM LOAD REPORT from: C:\Users\lihaodong\.cache\modelscope\hub\models\Qwen\Qwen3-1___7B
Key                                      | Status  | 
-----------------------------------------+---------+-
model.layers.{0...27}.self_attn.rot_mat  | MISSING | 
model.layers.{0...27}.self_attn.codebook | MISSING | 
model.layers.{0...27}.self_attn.proj_mat | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


配置: Baseline fp16 KV
耗时: 3.21s
峰值显存: 3319.92 MB
输出文本:
**核心观点：**  
TurboQuant 采用两阶段量化（MSE + QJL）对 KV cache 进行压缩，尽量在压缩后的表示上直接完成 attention 计算，以减少长上下文推理中的显存占用。

**3 条简明结论：**  
1. TurboQuant 通过量化技术压缩 KV cache，降低显存占用。  
2. 采用 MSE 和 QJL 两阶段量化，提升压缩效率。  
3. 在压缩表示上直接完成 attention 计算，优化长上下文推理性能。
----------------------------------------------------------------------------------------------------
配置: TurboQuant 3-bit KV
耗时: 2.94s
峰值显存: 3333.65 MB
输出文本:
**核心观点：**  
TurboQuant 采用两阶段量化（MSE + QJL）对 KV cache 进行压缩，尽量在压缩后的表示上直接完成 attention 计算，以减少长上下文推理中的显存占用。

**3 条简明结论：**  
1. TurboQuant 通过量化技术压缩 KV cache，降低显存占用。  
2. 采用 MSE 和 QJL 两阶段量化，提升压缩效率。  
3. 在压缩表示上直接完成 attention 计算，优化长上下文推理性能。
----------------------------------------------------------------------------------------------------


In [3]:
long_prompt, long_tokens = build_long_prompt(min_tokens=4096)
print(f"长输入 token 数: {long_tokens}")

baseline_long = run_long_case("Baseline fp16 KV", {"use_turboquant": False}, long_prompt, max_new_tokens=8)
turbo_long = run_long_case(
    "TurboQuant 3-bit KV",
    {"use_turboquant": True, "turboquant_bits": 3, "turboquant_block_size": 256},
    long_prompt,
    max_new_tokens=8,
)

print("=" * 100)
print(f"{'配置':<24} {'Prefill(s)':<12} {'Gen(s)':<12} {'PrefillMem(MB)':<16} {'PeakMem(MB)':<14}")
print("-" * 100)
for result in [baseline_long, turbo_long]:
    print(
        f"{result['description']:<24} {result['prefill_time_sec']:<12.2f} {result['gen_time_sec']:<12.2f} "
        f"{result['prefill_mem_mb']:<16.2f} {result['peak_mem_mb']:<14.2f}"
    )
print("=" * 100)

长输入 token 数: 4096


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3ForCausalLM LOAD REPORT from: C:\Users\lihaodong\.cache\modelscope\hub\models\Qwen\Qwen3-1___7B
Key                                      | Status  | 
-----------------------------------------+---------+-
model.layers.{0...27}.self_attn.rot_mat  | MISSING | 
model.layers.{0...27}.self_attn.codebook | MISSING | 
model.layers.{0...27}.self_attn.proj_mat | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


配置                       Prefill(s)   Gen(s)       PrefillMem(MB)   PeakMem(MB)   
----------------------------------------------------------------------------------------------------
Baseline fp16 KV         0.67         0.82         4930.73          6348.91       
TurboQuant 3-bit KV      2.22         7.61         4588.34          4619.82       


小文本时，KV cache 太小
     短输入下真正能省下来的 KV 显存本来就很有限，往往只有几十 MB 甚至更少。
     但 TurboQuant 会额外引入一些常驻数据和中间张量
     所以在短文本下：
  - 省下来的 KV 不够多
  - 额外开销反而更显眼
  - 最终就可能看到显存略微上升

更直观地说：
  - 短文本：KV收益 < TurboQuant额外开销
  - 长文本：KV收益 >> TurboQuant额外开销
